<a href="https://colab.research.google.com/github/anantasit0102-cmd/LAB-/blob/main/Project%20/2%22Thai%20Voice%20%22_2264_2297.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

+++สามารถทดลอง RUN ได้เลยนะครับผมได้บันทึกภาพผลแล้วเพราะผมไม่สามารถบันทึกไฟล์พร้อมผลรันได้ครับ

In [ ]:
import os
import time
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import Audio, display, HTML, clear_output
from google.colab.output import eval_js
from base64 import b64decode

print("✅ เครื่องมือพร้อม!")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile('thai_speech_dataset_2264_2297.zip', 'r') as zip_ref:
    zip_ref.extractall()

!ls

In [ ]:
import os
import pathlib
import numpy as np
from IPython.display import Audio, display

data_dir = pathlib.Path(".")

if not data_dir.exists():
    raise Exception(" path ผิด ลองเช็ค !ls")

commands = np.array([
    d for d in os.listdir(data_dir)
    if os.path.isdir(data_dir / d)and d not in ['.config', '__MACOSX', 'data', 'mini_speech_commands', 'sample_data']
])

print(" Classes:", commands)

sample_word = commands[0]

files = os.listdir(data_dir / sample_word)
sample_audio_path = data_dir / sample_word / files[0]

display(Audio(sample_audio_path, rate=16000))

In [ ]:
y, sr = librosa.load(sample_audio_path, sr=16000)

S = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_mels=64,
    fmax=8000
)

S_dB = librosa.power_to_db(S, ref=np.max)

plt.figure(figsize=(10,4))
librosa.display.specshow(S_dB, x_axis='time', y_axis='mel', sr=sr)
plt.colorbar()
plt.title("Mel-Spectrogram")
plt.show()

In [ ]:
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=data_dir,
    batch_size=16,
    validation_split=0.2,
    seed=0,
    output_sequence_length=16000,
    subset='both',
    class_names=commands.tolist()
)

label_names = np.array(train_ds.class_names)
num_classes = len(label_names)

print("Classes:", label_names)
print("Number of classes:", num_classes)

def get_spectrogram(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    spec = tf.signal.stft(audio, frame_length=255, frame_step=128)
    spec = tf.abs(spec)
    spec = tf.expand_dims(spec, -1)
    return spec, label

train_ds = train_ds.map(get_spectrogram).cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(get_spectrogram).cache().prefetch(tf.data.AUTOTUNE)

for spec, label in train_ds.take(1):
    input_shape = spec.shape[1:]

model = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Resizing(32, 32),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['train','val'])
plt.title("Accuracy")
plt.show()

In [ ]:
AUDIO_HTML = """
<script>
var my_btn = document.createElement("BUTTON");
my_btn.innerText = "อัดเสียง";
document.body.appendChild(my_btn);

var base64data = 0;

my_btn.onclick = async function(){
 let stream = await navigator.mediaDevices.getUserMedia({audio:true});
 let rec = new MediaRecorder(stream);
 rec.ondataavailable = e=>{
  let r=new FileReader();
  r.readAsDataURL(e.data);
  r.onloadend=()=>base64data=r.result;
 };
 rec.start();
 setTimeout(()=>rec.stop(),1500);
};

function getAudio(){
 return new Promise(r=>{
  let t=setInterval(()=>{
   if(base64data!=0){clearInterval(t);r(base64data);}
  },100);
 });
}
</script>
"""

display(HTML(AUDIO_HTML))
data = eval_js("getAudio()")

binary = b64decode(data.split(',')[1])
open("test.webm","wb").write(binary)

!ffmpeg -y -i test.webm -ac 1 -ar 16000 test.wav -loglevel quiet

audio, sr = librosa.load("test.wav", sr=16000)

# 1 วินาที = 16000
audio = audio[:16000]
if len(audio)<16000:
    audio = np.pad(audio,(0,16000-len(audio)))

tensor = tf.expand_dims(audio,-1)
spec,_ = get_spectrogram(tensor,0)
spec = tf.expand_dims(spec,0)

pred = model.predict(spec)
result = label_names[np.argmax(pred[0])]

print("AI ทำนาย:", result)